In [10]:
import numpy as np

# Input
n = int(input("Enter number of traffic signals: "))

traffic = []

for i in range(n):
    v = int(input(f"Enter vehicles at Signal {i+1}: "))
    traffic.append(v)

traffic = np.array(traffic)

# PSO parameters
particles = 20
iterations = 100
w = 0.5
c1 = 1.5
c2 = 1.5

total_time = 120
min_time = 10
max_time = 60


# Fitness function
def fitness(position):
    # Estimated waiting time in seconds
    waiting_time = traffic * (120 - position) / 120

    # Penalty if total green time is not 120
    penalty = abs(np.sum(position) - total_time) * 10

    return np.mean(waiting_time) + penalty


# Initialize particles
positions = np.random.uniform(
    min_time, max_time, (particles, n)
)

# Make total green time = 120
positions = (
    positions / positions.sum(axis=1, keepdims=True)
    * total_time
)

velocities = np.random.uniform(-2, 2, (particles, n))


# Personal best
pbest = positions.copy()
pbest_score = np.array([fitness(p) for p in positions])


# Global best
index = np.argmin(pbest_score)
gbest = pbest[index].copy()
gbest_score = pbest_score[index]


# PSO iterations
for iteration in range(iterations):

    for i in range(particles):

        r1 = np.random.rand(n)
        r2 = np.random.rand(n)

        # Update velocity
        velocities[i] = (
            w * velocities[i]
            + c1 * r1 * (pbest[i] - positions[i])
            + c2 * r2 * (gbest - positions[i])
        )

        # Update position
        positions[i] += velocities[i]

        # Keep green time within limits
        positions[i] = np.clip(
            positions[i],
            min_time,
            max_time
        )

        # Make total green time = 120
        positions[i] = (
            positions[i]
            / positions[i].sum()
            * total_time
        )

        # Calculate fitness
        score = fitness(positions[i])

        # Update pBest
        if score < pbest_score[i]:
            pbest[i] = positions[i].copy()
            pbest_score[i] = score

        # Update gBest
        if score < gbest_score:
            gbest = positions[i].copy()
            gbest_score = score


# Output
print("\n----- PSO RESULT -----")

for i in range(n):
    print(
        f"Signal {i+1}: "
        f"{gbest[i]:.2f} seconds green time"
    )

print(f"\nTotal Green Time: {gbest.sum():.2f} seconds")

print(
    f"Estimated Average Waiting Time: "
    f"{gbest_score:.2f} seconds"
)

Enter number of traffic signals: 4
Enter vehicles at Signal 1: 30
Enter vehicles at Signal 2: 70
Enter vehicles at Signal 3: 20
Enter vehicles at Signal 4: 30

----- PSO RESULT -----
Signal 1: 13.33 seconds green time
Signal 2: 80.00 seconds green time
Signal 3: 13.33 seconds green time
Signal 4: 13.33 seconds green time

Total Green Time: 120.00 seconds
Estimated Average Waiting Time: 23.61 seconds
